<a href="https://colab.research.google.com/github/sanjeevchikkam/Medicinemapping/blob/main/MedicineMapping(Fuzzy%20matching).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Start the project

## Install and import libraries

In [ ]:
!pip install rapidfuzz
import pandas as pd
import re
from rapidfuzz import process, fuzz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 84.1 MB/s eta 0:00:00


### upload the pharmacy and master medicine id and name

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving pharmacy_medicine.csv to pharmacy_medicine.csv
Saving master_medicine.csv to master_medicine.csv


In [ ]:
pharmacy = pd.read_csv("pharmacy_medicine.csv")
master = pd.read_csv("master_medicine.csv")

In [ ]:
pharmacy["medicine_name"]

,medicine_name
0,Paracetamol 500mg Tab
1,Paracetamol 650mg Tablet
2,Amoxycillin 250mg Cap
3,Amoxycillin 500 mg Capsule
4,Ibuprofen 200mg Tab
5,Ibuprofen 400mg Tablet
6,Cetrizine 10mg Tab
7,Levocetrizine 5 mg Tablet
8,Metformin 500mg Tab
9,Metformin 1g Tablet


# Write the Normalize text

In [ ]:
def normalize_text(text):

    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [ ]:
pharmacy["normalized_name"] = pharmacy["medicine_name"].apply(normalize_text)
master["normalized_name"] = master["medicine_name"].apply(normalize_text)

In [ ]:
master["normalized_name"]

,normalized_name
0,paracetamol 500 mg tablet
1,paracetamol 650 mg tablet
2,amoxicillin 250 mg capsule
3,amoxicillin 500 mg capsule
4,ibuprofen 200 mg tablet
5,ibuprofen 400 mg tablet
6,cetirizine 10 mg tablet
7,levocetirizine 5 mg tablet
8,metformin 500 mg tablet
9,metformin 1000 mg tablet


In [ ]:
pharmacy["normalized_name"]

,normalized_name
0,paracetamol 500mg tablet
1,paracetamol 650mg tabletlet
2,amoxycillin 250mg capsule
3,amoxycillin 500 mg capsulesule
4,ibuprofen 200mg tablet
5,ibuprofen 400mg tabletlet
6,cetrizine 10mg tablet
7,levocetrizine 5 mg tabletlet
8,metformin 500mg tablet
9,metformin 1g tabletlet


### replace tab -> tablet

In [ ]:
abbrev = {
    r"\btab\b": "tablet",
    r"\bcap\b": "capsule",
    r"\binj\b": "injection",
    r"\bsyp\b": "syrup"
}

pharmacy["normalized_name"] = pharmacy["normalized_name"].replace(abbrev, regex=True)

In [ ]:
abbrev = {
    "tab": "tablet",
    "cap": "capsule",
    "inj": "injection",
    "syp": "syrup"
}

for k, v in abbrev.items():
    pharmacy["normalized_name"] = pharmacy["normalized_name"].str.replace(
        rf"\b{k}\b", v, regex=True
    )

In [ ]:
pharmacy["normalized_name"]

,normalized_name
0,paracetamol 500mg tablet
1,paracetamol 650mg tabletlet
2,amoxycillin 250mg capsule
3,amoxycillin 500 mg capsulesule
4,ibuprofen 200mg tablet
5,ibuprofen 400mg tabletlet
6,cetrizine 10mg tablet
7,levocetrizine 5 mg tabletlet
8,metformin 500mg tablet
9,metformin 1g tabletlet


In [ ]:
master["normalized_name"]

,normalized_name
0,paracetamol 500 mg tablet
1,paracetamol 650 mg tablet
2,amoxicillin 250 mg capsule
3,amoxicillin 500 mg capsule
4,ibuprofen 200 mg tablet
5,ibuprofen 400 mg tablet
6,cetirizine 10 mg tablet
7,levocetirizine 5 mg tablet
8,metformin 500 mg tablet
9,metformin 1000 mg tablet


In [ ]:
master_names = master["normalized_name"].tolist()

## Match Master

##Fuzzy match

In [ ]:
def match_master(medicine):

    match, score, index = process.extractOne(
        medicine,
        master_names,
        scorer=fuzz.token_sort_ratio
    )

    master_id = master.iloc[index]["master_id"]

    return pd.Series([master_id, match, score])

## calling and matching the pharmacy medi name

In [ ]:
pharmacy[["master_id","matched_name","score"]] = pharmacy["normalized_name"].apply(match_master)

In [ ]:
final_inventory = pharmacy[[
    "pharmacy_id",
    "medicine_name",
    "master_id",
    "matched_name",
    "score"
]]

final_inventory = final_inventory.rename(columns={
    "medicine_name":"pharmacy_name"
})

print(final_inventory)

   pharmacy_id                 pharmacy_name master_id  \
0         P001         Paracetamol 500mg Tab      M001   
1         P002      Paracetamol 650mg Tablet      M002   
2         P003         Amoxycillin 250mg Cap      M003   
3         P004    Amoxycillin 500 mg Capsule      M004   
4         P005           Ibuprofen 200mg Tab      M005   
5         P006        Ibuprofen 400mg Tablet      M006   
6         P007            Cetrizine 10mg Tab      M007   
7         P008     Levocetrizine 5 mg Tablet      M008   
8         P009           Metformin 500mg Tab      M009   
9         P010           Metformin 1g Tablet      M010   
10        P011             Amlodipin 5mg Tab      M011   
11        P012        Amlodipine 10mg Tablet      M012   
12        P013         Atorvastatin 10mg Tab      M013   
13        P014            Atorva 20mg Tablet      M014   
14        P015        Azithromycin 250mg Tab      M015   
15        P016          Azithro 500mg Tablet      M016   
16        P017

# To Download

In [ ]:
final_inventory.to_csv("pharmacy_inventory.csv", index=False)

## Final Inventory csv file

In [ ]:
from google.colab import files
files.download("pharmacy_inventory.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>